# ACS 2024 Income Model — Experiment 3 (high-cardinality grouping / coarsening)

A working scaffold derived from **`ML_Model_Building_Template.md`**, wired to **`acs2024_income.csv`** — the rich 18-feature ACS 2024 extract produced by `acs2024_income_extraction.ipynb`. Run top-to-bottom.

**Dataset:** `acs2024_income.csv` — 339,727 employed working-age adults, socio-economic predictors from the 2024 ACS PUMS (1-Year).

| Column | Meaning | Kind |
|---|---|---|
| `AGEP` | Age in years | continuous |
| `WKHP` | Usual hours worked per week | continuous |
| `WKWN` | Weeks worked in past 12 months | continuous |
| `COW` | Class of worker (employment sector) | categorical |
| `SCHL` | Education level (ordinal code) | categorical |
| `MAR` | Marital status | categorical |
| `OCCP` | Occupation code | categorical |
| `INDP` | Industry code | categorical |
| `POBP` | Place of birth | categorical |
| `RELSHIPP` | Relationship to householder | categorical |
| `SEX` | Sex | categorical |
| `RAC1P` | Race group | categorical |
| `HISP` | Hispanic origin | categorical |
| `CIT` | Citizenship status | categorical |
| `NATIVITY` | Nativity (native / foreign-born) | categorical |
| `ENG` | English-speaking ability | categorical |
| `DIS` | Disability | categorical |
| `ST` | State (FIPS code) | categorical |
| **`WAGP`** | **Target — wages/salary income (scaled)** | continuous |

| Stage | Name | Key concern |
|---|---|---|
| 1 | Environment Setup | Reproducibility |
| 2 | Data Loading & Preview | Understand structure & quality |
| 3 | Data Cleaning | Missing values -> encoding -> imbalance |
| 4 | Graphical EDA | Distributions & relationships |
| 5 | Feature Set (grouping) | Coarsen high-card codes: rollup + rare-pool |
| 6 | Train / Val / Test Split | Stratified, no leakage |
| 7 | Feature Scaling | Fit on train only |
| 8 | Model Architecture | Algorithm choice & hyperparameters |
| 9 | Training | Cross-validation or epoch-based |
| 10 | Learning Curves | Diagnose over/underfitting |
| 11 | Evaluation | Training + test metrics |
| 12 | Save & Inference | Persistence & deployment |

> **Task type:** **regression** — predict `WAGP` from the remaining 18 features. The categorical columns are integer-coded ACS values (`OCCP`, `INDP`, `POBP`, `ST` are high-cardinality), which suits native-categorical gradient-boosted trees.
>
> **Models:** Stages 8-12 benchmark **ten regressors grouped by family** — linear baselines (Ridge, Linear, Lasso), boosted trees (XGBoost, LightGBM, CatBoost), bagging/extra boosting (RandomForest, ExtraTrees, HistGradientBoosting), and a stacking blend of the three boosters. **This is EXPERIMENT 3:** it keeps the raw feature set but **coarsens the four high-cardinality nominal codes** (`OCCP` 525, `INDP` 257, `POBP` 222, `ST` 51 levels) into low-cardinality groups — a **semantic rollup** of *similar* classes (Stage 5b) plus **rare-level pooling** of *low-sample* classes (Stage 6b) — cutting ~1,055 category levels down to ~50. Unlike Experiment 2 (which *added* numeric target/frequency encodings and kept the raw codes), Experiment 3 *replaces* the raw codes with the groups. Compare against `acs2024_base_model.ipynb` (raw codes) and `acs2024_exp2_model.ipynb` (current best) to see what grouping buys.

## Stage 1 — Environment Setup & Library Imports

Import everything upfront, set display options so nothing is truncated, and fix a global seed for reproducibility.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sbn

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Display options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
np.set_printoptions(threshold=10000)

## Stage 2 — Data Loading & Textual Preview

Load `acs2024_income.csv` and inspect it textually: shape, head/tail, schema, summary stats, and target distribution. Note any data-quality problems to fix in Stage 3.

In [ ]:
# Load dataset
dataset = pd.read_csv("acs2024_income.csv", header=0, index_col=None)

TARGET_COLUMN   = "WAGP"
FEATURE_COLUMNS = [c for c in dataset.columns if c != TARGET_COLUMN]
# Continuous vs. categorical (categoricals are integer-coded ACS values).
# OCCP, INDP, POBP, ST are high-cardinality -> prefer native-categorical models.
CONTINUOUS_COLS  = ["AGEP", "WKHP", "WKWN"]
CATEGORICAL_COLS = ["COW", "SCHL", "MAR", "OCCP", "INDP", "POBP", "RELSHIPP",
                    "SEX", "RAC1P", "HISP", "CIT", "NATIVITY", "ENG", "DIS", "ST"]

print("Shape:", dataset.shape, "\n")
print(dataset.head(), "\n")
print(dataset.tail(), "\n")

In [ ]:
# Structure, dtypes, missing counts, summary stats
print(dataset.info(), "\n")
print(dataset.describe(), "\n")
print("Missing values per column:\n", dataset.isna().sum(), "\n")

# Cardinality of categorical columns (flags high-cardinality features for Stage 8)
print("Unique values per categorical column:")
print(dataset[CATEGORICAL_COLS].nunique().sort_values(ascending=False), "\n")

# Target distribution (regression -> describe + histogram)
print(dataset[TARGET_COLUMN].describe())
dataset[TARGET_COLUMN].hist(bins=60, figsize=(10, 4))
plt.title("WAGP (wages) distribution -- note the right skew"); plt.show()

### Stage 2 — Data-quality notes

- **Shape:** 339,727 rows x 19 columns (18 features + `WAGP` target).
- **No missing values:** the extraction step already filtered and sentinel-filled, so Stage 3a is a guard/no-op here.
- **Categoricals are integer-coded:** no string encoding needed. `OCCP`, `INDP`, `POBP`, and `ST` are high-cardinality (hundreds of levels) -> favour native-categorical learners (CatBoost / XGBoost / LightGBM) over one-hot.
- **Target `WAGP` is already scaled** to roughly `[0, 0.92]` (not raw dollars) and is **right-skewed** -> `log1p` transform in Stage 3d; metrics are converted back to dollars (x1e6) in Stage 11.

## Stage 3 — Data Cleaning & Problem Fixing

Fix problems **before** encoding/transformation, in the template order **missing values -> categorical encoding -> class imbalance**, plus a few dataset-specific checks. For `acs2024_income.csv` most template steps are guards/no-ops (no nulls, already integer-coded, regression target), so each cell below keeps the **recommended** path active and the **alternatives commented** so the full menu is visible.

**The menu**

| Sub-stage | Recommended (active) | Alternatives (commented) |
|---|---|---|
| 3a Missing values | guard (no nulls present) | KNN / median impute; sentinel level |
| 3b Categorical encoding | keep native, enforce `int` codes | one-hot (low-card only); ordinal (`SCHL`/`ENG`); frequency/target encode (high-card); coarsen to ACS rollups |
| 3c Class imbalance | N/A (continuous target) | bracket `WAGP` + SMOTE/undersample (classification variant) |
| 3d Dataset-specific | duplicate check; `log1p` target | winsorize top-coded tail |

In [ ]:
# 3a -- Missing values. acs2024_income.csv has NONE, so this is a guard/no-op.
# (The extractor already sentinel-filled non-applicable OCCP/INDP with -1, i.e.
#  "no occupation/industry" is its own category rather than a NaN.)
from sklearn.impute import KNNImputer

cols_with_nulls = [c for c in dataset.columns if dataset[c].isna().any()]
if cols_with_nulls:
    cont_nulls = [c for c in cols_with_nulls if c in CONTINUOUS_COLS]
    cat_nulls  = [c for c in cols_with_nulls if c in CATEGORICAL_COLS]
    if cont_nulls:                                   # continuous -> KNN (or median)
        dataset[cont_nulls] = KNNImputer(n_neighbors=5).fit_transform(dataset[cont_nulls])
    for c in cat_nulls:                              # categorical -> dedicated sentinel
        dataset[c] = dataset[c].fillna(-1)

print("Columns with nulls:", cols_with_nulls)
print("Remaining missing values:", int(dataset.isna().sum().sum()))

In [ ]:
# 3b -- Categorical handling. Columns are ALREADY integer-coded ACS values, so no
# string encoding is needed; the real decision is how DOWNSTREAM models treat them.
HIGH_CARD = ["OCCP", "INDP", "POBP", "ST"]                 # 525 / 257 / 222 / 51 levels
LOW_CARD  = [c for c in CATEGORICAL_COLS if c not in HIGH_CARD]

# === RECOMMENDED: keep native categorical, just enforce integer codes ==========
# CatBoost consumes these directly via cat_features; XGBoost/LightGBM via category dtype.
for c in CATEGORICAL_COLS:
    dataset[c] = dataset[c].astype(int)
print("Categoricals enforced as int. High-cardinality levels:",
      {c: int(dataset[c].nunique()) for c in HIGH_CARD})

# === ALTERNATIVE 1 -- One-hot the LOW-cardinality columns only =================
# Good for linear / distance models. NEVER one-hot OCCP/INDP/POBP (-> 1000+ cols).
# dataset = pd.get_dummies(dataset, columns=LOW_CARD, drop_first=True)

# === ALTERNATIVE 2 -- Ordinal encode the genuinely ORDERED columns =============
# Only SCHL (education) and ENG (English ability) have a meaningful order; they are
# already monotonic codes, so use this only if starting from raw string categories.
# from sklearn.preprocessing import OrdinalEncoder
# dataset[["SCHL", "ENG"]] = OrdinalEncoder().fit_transform(dataset[["SCHL", "ENG"]])

# === ALTERNATIVE 3 -- Frequency / target encoding for HIGH-cardinality =========
# For models WITHOUT native categorical support (sklearn RF, linear). Frequency
# encoding is leak-free; target encoding MUST be fit inside CV folds (Stage 6+).
# for c in HIGH_CARD:
#     dataset[c + "_freq"] = dataset[c].map(dataset[c].value_counts(normalize=True))

# === ALTERNATIVE 4 -- Coarsen high-cardinality codes to ACS rollups ============
# dataset["OCCP_grp"] = (dataset["OCCP"] // 100).astype(int)    # ~major occ groups
# dataset["INDP_grp"] = (dataset["INDP"] // 1000).astype(int)   # ~sectors

In [ ]:
# 3c -- Class imbalance: NOT APPLICABLE (WAGP is a continuous regression target).
# Only relevant for a CLASSIFICATION variant where WAGP is binned into brackets:
# brackets = pd.qcut(dataset[TARGET_COLUMN], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
# then rebalance the resulting classes (SVMSMOTE / RandomUnderSampler).
print("Regression target -- no resampling applied.")

In [ ]:
# 3d -- Dataset-specific checks beyond the template.

# Duplicate rows: PUMS can legitimately repeat identical feature rows -> inspect, keep.
print("Exact duplicate rows:", int(dataset.duplicated().sum()))

# Outliers / top-coding: Census top-codes very high wages, compressing the right tail.
# WAGP here is already scaled to ~[0, 0.92]. OPTIONAL winsorize at the 99.5th pct:
# cap = dataset[TARGET_COLUMN].quantile(0.995)
# dataset[TARGET_COLUMN] = dataset[TARGET_COLUMN].clip(upper=cap)

# Target transform (right-skew): build a log1p target as a SEPARATE series (do NOT
# add it as a feature column). Invert with expm1 before reporting metrics in Stage 11.
y_log = np.log1p(dataset[TARGET_COLUMN])
print("\nlog1p(WAGP) summary:")
print(y_log.describe())

## Stage 4 — Graphical Data Exploration (EDA)

Visualise distributions and relationships before engineering features. The skewed continuous histograms justify the target transform; the grouped-median bars and the focused scatter confirm the headline socio-economic signal (education, hours, sex) is present.

In [ ]:
# Distribution of the continuous features + target. (Categorical columns are integer
# CODES, so a numeric histogram of them is not meaningful -- a few key ones are shown
# as grouped medians below instead.)
dataset[CONTINUOUS_COLS + [TARGET_COLUMN]].hist(bins=50, figsize=(12, 8))
plt.tight_layout(); plt.show()

In [ ]:
# Pairwise scatter matrix on the continuous features + target (sampled for speed).
from pandas.plotting import scatter_matrix

attrs = CONTINUOUS_COLS + [TARGET_COLUMN]
scatter_matrix(dataset[attrs].sample(5000, random_state=RANDOM_STATE),
               alpha=0.4, figsize=(11, 9), diagonal="kde")
plt.suptitle("Scatter matrix -- continuous features vs. WAGP", y=1.01); plt.show()

In [ ]:
# Focused scatter: education vs. usual hours worked, coloured by wage.
dataset.sample(5000, random_state=RANDOM_STATE).plot(
    kind="scatter", x="SCHL", y="WKHP", c=TARGET_COLUMN, cmap="jet",
    colorbar=True, grid=True, figsize=(13, 8), alpha=0.5)
plt.title("SCHL (education) vs WKHP (hours) coloured by WAGP"); plt.show()

In [ ]:
# Signal check: median wage by a few key categoricals (codes ordered on the x-axis).
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
dataset.groupby("SCHL")[TARGET_COLUMN].median().plot(kind="bar", ax=ax[0])
ax[0].set_title("Median WAGP by education (SCHL)")
dataset.groupby("SEX")[TARGET_COLUMN].median().plot(kind="bar", ax=ax[1])
ax[1].set_title("Median WAGP by sex (SEX)")
dataset.groupby("COW")[TARGET_COLUMN].median().plot(kind="bar", ax=ax[2])
ax[2].set_title("Median WAGP by class of worker (COW)")
plt.tight_layout(); plt.show()

## Stage 5 — Feature Set (Experiment 3: coarsen, don't encode)

The baseline showed that **keeping all raw features beats selection**, and Experiment 2 showed that the high-cardinality codes (`OCCP`/`INDP`/`POBP`/`ST`) carry most of the signal. Experiment 3 attacks those same codes from a different angle than exp2: instead of *adding* numeric target/frequency encodings, it **reduces their cardinality by grouping**, investigating two complementary methodologies:

- **5b — Semantic rollup (group *similar* classes):** map each fine code to its published ACS/Census parent block — SOC occupation groups, NAICS industry sectors, birthplace regions, Census state regions. Leak-free (no target), so it runs before the split and **replaces** the raw codes.
- **6b — Rare-level pooling (group *low-sample* classes):** after the split, collapse categorical levels with too few training rows into a shared "Other" bucket (leak-safe, train counts only).

No interaction terms and no Select-K-Best (those are exp1/exp2). We still rank the grouped set by mutual information for insight only.

In [ ]:
# 5a -- Feature identification. Experiment 3 starts from the raw feature columns; the
# high-cardinality codes get REPLACED by semantic groups in 5b. y_sel is the log1p target
# (Stage 3d); trgt_y keeps raw WAGP for dollar-scale reporting later.
feats_X = dataset[FEATURE_COLUMNS].copy()
trgt_y  = pd.DataFrame(dataset[[TARGET_COLUMN]])
y_sel   = y_log
print("feats_X (raw 18 features):", feats_X.shape, "| trgt_y:", trgt_y.shape)

In [ ]:
# 5b -- Feature GROUPING, methodology 1: group SIMILAR classes via a SEMANTIC ROLLUP.
# Replace the four high-cardinality nominal codes with low-cardinality groups, using the
# published ACS / Census code-block structure. Pure code->group lookups (the target is
# never touched) -> leak-free, so this runs before the split.
HIGH_CARD = ["OCCP", "INDP", "POBP", "ST"]

# Lower bound of each Census code block; np.searchsorted assigns every code its block index.
OCCP_BOUNDS = [10, 500, 800, 1005, 1300, 1600, 2001, 2100, 2205, 2600, 3000, 3601,
               3700, 4000, 4200, 4330, 4700, 5000, 6005, 6200, 6800, 7000, 7700,
               9005, 9800]              # SOC major occupation groups (MGR .. MIL)
INDP_BOUNDS = [170, 370, 570, 770, 1070, 4070, 4670, 6070, 6470, 6870, 7270, 7860,
               7970, 8560, 8770, 9370, 9670]     # NAICS industry sectors (AGR .. MIL)
POBP_BOUNDS = [1, 100, 200, 300, 400, 500]       # US-born + Europe/Asia/Americas/Africa/Oceania
ST_REGION = {                                    # state FIPS -> 4 Census regions
    9: 1, 23: 1, 25: 1, 33: 1, 44: 1, 50: 1, 34: 1, 36: 1, 42: 1,                  # Northeast
    17: 2, 18: 2, 26: 2, 39: 2, 55: 2, 19: 2, 20: 2, 27: 2, 29: 2, 31: 2, 38: 2, 46: 2,  # Midwest
    10: 3, 11: 3, 12: 3, 13: 3, 24: 3, 37: 3, 45: 3, 51: 3, 54: 3,
    1: 3, 21: 3, 28: 3, 47: 3, 5: 3, 22: 3, 40: 3, 48: 3,                          # South
    2: 4, 4: 4, 6: 4, 8: 4, 15: 4, 16: 4, 30: 4, 32: 4, 35: 4, 41: 4, 49: 4, 53: 4, 56: 4}  # West

def _to_group(s, bounds):
    """Map each integer code to the index (1..len) of the code block it falls in."""
    return np.searchsorted(bounds, s.to_numpy(), side="right").astype(int)

feats_X["OCCP_grp"] = _to_group(feats_X["OCCP"], OCCP_BOUNDS)
feats_X["INDP_grp"] = _to_group(feats_X["INDP"], INDP_BOUNDS)
feats_X["POBP_grp"] = _to_group(feats_X["POBP"], POBP_BOUNDS)
feats_X["ST_grp"]   = feats_X["ST"].map(ST_REGION).fillna(0).astype(int)

GROUP_MAP      = {"OCCP": "OCCP_grp", "INDP": "INDP_grp", "POBP": "POBP_grp", "ST": "ST_grp"}
GROUPED_COLS   = list(GROUP_MAP.values())
GENERATED_COLS = GROUPED_COLS                    # populate the template's bookkeeping name

# Drop the raw high-card codes -> the models see ONLY the coarse groups (the experiment).
feats_X = feats_X.drop(columns=HIGH_CARD, errors="ignore")

# Keep the notebook's categorical bookkeeping in sync: swap raw high-card -> grouped names.
CATEGORICAL_COLS = [GROUP_MAP.get(c, c) for c in CATEGORICAL_COLS]

print("Cardinality collapse (raw level count -> grouped level count):")
for raw, grp in GROUP_MAP.items():
    print(f"  {raw:5s} {dataset[raw].nunique():4d}  ->  {grp:9s} {feats_X[grp].nunique():3d}")
print("Total high-card levels:", sum(int(dataset[c].nunique()) for c in HIGH_CARD),
      "->", sum(int(feats_X[c].nunique()) for c in GROUPED_COLS))
print("feats_X now:", feats_X.shape)

In [ ]:
# 5c -- Feature selection: SKIPPED (no Select-K-Best -- the baseline showed selection
# hurts these trees). Rank the GROUPED feature set by mutual information for INSIGHT only.
from sklearn.feature_selection import mutual_info_regression

ALL_FEATURES  = list(feats_X.columns)
discrete_mask = [c in CATEGORICAL_COLS for c in ALL_FEATURES]   # grouped codes are discrete

samp = feats_X.sample(min(20000, len(feats_X)), random_state=RANDOM_STATE)
mi_scores = pd.Series(
    mutual_info_regression(samp, y_sel.loc[samp.index],
                           discrete_features=discrete_mask, random_state=RANDOM_STATE),
    index=ALL_FEATURES).sort_values(ascending=False)
selected = ALL_FEATURES
print("Mutual information with log(WAGP) [grouped feature set]:")
print(mi_scores.round(4))
print("Experiment 3 keeps all", len(selected), "features (4 raw codes replaced by groups).")

In [ ]:
# 5c (heatmap) -- multicollinearity check on the RAW numeric features + target only.
numeric_cols = CONTINUOUS_COLS                   # no generated features in the baseline
corr = pd.concat([feats_X[numeric_cols], y_log.rename("WAGP_log")], axis=1).corr("pearson")

plt.figure(figsize=(7, 6))
sbn.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, square=True)
plt.title("Pearson correlation -- raw numeric features + log target"); plt.show()

In [ ]:
# Finalise the grouped feature set: continuous + low-card categoricals + the 4 coarse
# groups (raw OCCP/INDP/POBP/ST dropped). Rare-level pooling is applied after the split (6b).
useful_feats_X = feats_X[selected].copy()
print("Grouped feature set:", list(useful_feats_X.columns))
print("useful_feats_X:", useful_feats_X.shape)

## Stage 6 — Train / Validation / Test Split

Regression target -> a plain shuffled `train_test_split` (an optional `qcut`-bin stratification is shown to balance income *ranges*). A validation slice is carved from train for early stopping. Targets are `log1p(WAGP)`; the `*_raw` series recover the (scaled) WAGP axis via `expm1`, then Stage 11 multiplies by 1e6 for dollars.

In [ ]:
from sklearn.model_selection import train_test_split

X = useful_feats_X
y = y_sel                                   # = y_log, index-aligned with X

# Categorical subset that survived selection -> native categorical handling in Stage 8.
CAT_FEATURES = [c for c in X.columns if c in CATEGORICAL_COLS]
print("Categorical features carried into the models:", CAT_FEATURES)

# Primary 80/20 train-test split (shuffled).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, shuffle=True)

# Optional: balance income RANGES across splits instead (uncomment to stratify):
# strat_bins = pd.qcut(y, q=10, labels=False)
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=strat_bins)

# Carve a validation slice out of train (early stopping).
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE, shuffle=True)

# Raw-axis targets aligned to each split (expm1 inverts the log1p), for Stage 11.
y_train_raw = np.expm1(y_train)
y_val_raw   = np.expm1(y_val)
y_test_raw  = np.expm1(y_test)

print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)

## Stage 6b — Rare-level pooling (group low-sample classes)

The Stage-5b rollup grouped *similar* classes; this second methodology groups *low-sample* classes. Any categorical level seen in **fewer than `MIN_COUNT` training rows** is collapsed into a shared **"Other" (`-999`)** bucket. Counts use the **training split only** (a feature-frequency rule — the target is never used → leak-free); val/test are remapped with the train-derived keep-lists, so a level that is rare *or unseen* in train also lands in "Other".

After the semantic rollup the four engineered groups are already well populated, so at `MIN_COUNT = 30` pooling only trims the last sparse codes in the low-cardinality columns (a few rare `RELSHIPP` / `HISP` levels). It is the same mechanism that *would* have absorbed the raw `OCCP`/`POBP` long tails had we pooled instead of rolled up — shown here as the second, complementary grouping lever.

In [ ]:
# Stage 6b -- Rare-level pooling. Collapse categorical levels with < MIN_COUNT TRAIN rows
# into a shared "Other" bucket (-999). Train-only counts -> leak-free; map onto val/test.
MIN_COUNT  = 30
POOL_OTHER = -999

pooled = {}
for c in CAT_FEATURES:                       # CAT_FEATURES already holds the grouped names
    vc   = X_train[c].value_counts()
    keep = set(vc[vc >= MIN_COUNT].index)
    if len(keep) < X_train[c].nunique():     # only touch columns that actually have rare levels
        pooled[c] = keep
        for frame in (X_train, X_val, X_test):
            frame[c] = np.where(frame[c].isin(keep), frame[c], POOL_OTHER).astype(int)

print(f"Rare-level pooling (train count < {MIN_COUNT} -> {POOL_OTHER}):")
if pooled:
    for c, keep in pooled.items():
        print(f"  {c:9s}: kept {len(keep):3d} levels + Other")
else:
    print("  no column had rare levels at this threshold")
print("Categorical cardinality after grouping + pooling:",
      {c: int(X_train[c].nunique()) for c in CAT_FEATURES})

## Stage 7 — Feature Scaling / Normalisation

**Primary (tree) path: no scaling.** Gradient-boosted trees are scale-invariant, the target is already `log1p`-transformed, and scaling would corrupt the integer category codes. The commented **DL / linear branch** shows the leakage-safe pattern: fit on **train only**, transform **numeric features only**.

In [ ]:
from sklearn.preprocessing import QuantileTransformer, StandardScaler

# === PRIMARY (tree) path: NO scaling -- pass through ===========================
X_train_sc, X_val_sc, X_test_sc = X_train, X_val, X_test
y_train_sc, y_val_sc, y_test_sc = y_train, y_val, y_test
print("Tree path: features passed through unscaled.")

# === ALTERNATIVE (DL / linear branch): scale NUMERIC features only =============
# NUMERIC_COLS = [c for c in X_train.columns if c not in CAT_FEATURES]
# scaler_X = QuantileTransformer(output_distribution="normal", random_state=RANDOM_STATE)
# scaler_X.fit(X_train[NUMERIC_COLS])
# def _scale(df):
#     out = df.copy(); out[NUMERIC_COLS] = scaler_X.transform(df[NUMERIC_COLS]); return out
# X_train_sc, X_val_sc, X_test_sc = _scale(X_train), _scale(X_val), _scale(X_test)

## Stage 8 — Model Selection, Architecture & Compilation

Ten regressors are configured and benchmarked, grouped by family, all on the same selected features and the same `log1p(WAGP)` target:

**1. Linear baselines — Ridge / plain Linear / Lasso.** A floor for the comparison. They can't consume the integer ACS codes directly (those codes are *nominal* — a larger `OCCP` isn't "more"), so each is wrapped in a `Pipeline` whose `ColumnTransformer` **standardises the numeric features and one-hot encodes the categoricals**, fit on the training fold only (leakage-safe).

**2. Boosted trees — XGBoost / LightGBM / CatBoost.** Matched hyperparameters (`~2000` trees, `lr=0.05`, `max_depth=8`, L2, early stopping) so differences reflect the *algorithm*. All three ingest the categoricals **natively**, including high-cardinality `OCCP`/`INDP`/`ST`:
- **CatBoost** — `cat_features` (ordered boosting). **XGBoost** — `enable_categorical=True`. **LightGBM** — `categorical_feature=` (both need pandas `category` dtype).

**3. Bagging & extra boosting — RandomForest / ExtraTrees / HistGradientBoosting.** Contrasting tree paradigms (bagging vs. boosting) **without** native high-cardinality support — they read the integer codes as *ordinal*, so they're the "no native categoricals" comparison point. HistGB's native categorical mode caps at 255 categories, which `OCCP`(525)/`INDP`(257) exceed, so it too treats the codes as ordinal.

**4. Stacking ensemble — blend of XGBoost + LightGBM + CatBoost.** A `StackedGBM` holdout-blender: the three fitted boosters predict on the held-out validation split, and a linear meta-learner learns how to weight them. This is the one model most likely to edge past the individual boosters. (A holdout blend is used instead of full CV stacking because re-fitting three heavy boosters across CV folds on 217k rows is far more expensive for a marginal gain.)

In [ ]:
from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

cat_model = CatBoostRegressor(
    iterations=2000, learning_rate=0.05, depth=8, l2_leaf_reg=10,
    loss_function="RMSE", eval_metric="RMSE", random_state=RANDOM_STATE,
    early_stopping_rounds=100, verbose=400)

xgb_model = XGBRegressor(
    n_estimators=2000, learning_rate=0.05, max_depth=8, subsample=0.8,
    colsample_bytree=0.8, reg_lambda=10, tree_method="hist",
    enable_categorical=True, early_stopping_rounds=100,
    random_state=RANDOM_STATE, verbosity=0)

lgb_model = LGBMRegressor(
    n_estimators=2000, learning_rate=0.05, max_depth=8, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=10,
    random_state=RANDOM_STATE, verbose=-1)

print("Configured CatBoost, XGBoost, LightGBM.")
print("Categorical features:", CAT_FEATURES)

# --- Linear baselines: Ridge / Linear / Lasso --------------------------------
# Unlike the trees, linear models can't read the integer ACS codes directly: those
# codes are NOMINAL (a higher OCCP number isn't "more"), so they must be one-hot'd,
# and the numeric features must be standardised. A ColumnTransformer does both and is
# fit INSIDE each pipeline -> on the training fold only, so the split stays leak-free.
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NUMERIC_FOR_LINEAR = [c for c in X.columns if c not in CAT_FEATURES]   # continuous + generated
linear_pre = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FOR_LINEAR),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),     # sparse -> high-card OK
])

def make_linear(estimator):
    """Wrap a linear estimator with the shared scale + one-hot preprocessor."""
    return Pipeline([("prep", linear_pre), ("model", estimator)])

ridge_model  = make_linear(Ridge(alpha=1.0))
linear_model = make_linear(LinearRegression())
lasso_model  = make_linear(Lasso(alpha=1e-4, max_iter=3000, random_state=RANDOM_STATE))

print("\nConfigured Ridge, LinearRegression, Lasso (StandardScaler + OneHotEncoder pipeline).")
print("Numeric  ->", NUMERIC_FOR_LINEAR)
print("One-hot  ->", CAT_FEATURES)

# --- Bagging & extra-boosting baselines: RandomForest / ExtraTrees / HistGB ---
# None of these have native high-cardinality categorical support, so they read the
# integer ACS codes as ORDINAL numbers (a known limitation vs CatBoost/LightGBM). We
# still feed the plain integer frames -- one-hot would explode OCCP(525)/INDP(257) and
# is unnecessary for trees. This makes them a fair "no native categoricals" contrast.
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              HistGradientBoostingRegressor)

rf_model = RandomForestRegressor(
    n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE)

et_model = ExtraTreesRegressor(
    n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE)

# HistGB's native categorical mode caps at 255 categories/feature, but OCCP(525)/INDP(257)
# exceed that -> we let it treat the codes as ordinal and use its built-in early stopping
# (internal validation split carved from train).
hgb_model = HistGradientBoostingRegressor(
    max_iter=2000, learning_rate=0.05, max_depth=8, l2_regularization=10,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=50,
    random_state=RANDOM_STATE)

print("\nConfigured RandomForest, ExtraTrees, HistGradientBoosting (plain integer codes).")

# --- Stacking ensemble of the three boosters (XGB + LightGBM + CatBoost) ------
# Holdout BLEND (cheaper than full CV stacking on 217k x 3 heavy models): the three
# already-fitted boosters predict on the held-out VALIDATION split, and a linear
# meta-learner learns how to weight them. Leakage-safe -- bases train on X_train, the
# meta trains on X_val, and everything is finally scored on the untouched X_test.
class StackedGBM:
    """Blend XGBoost + LightGBM + CatBoost predictions with a linear meta-learner.

    predict() accepts the PLAIN integer frame and rebuilds the category-dtype copy that
    XGBoost/LightGBM need, casting with the TRAINING categories so the integer->code
    mapping stays consistent (unseen levels become NaN = missing, handled natively)."""
    def __init__(self, base_models, cat_dtypes, meta=None):
        self.base_models = base_models      # list of (name, fitted_model, needs_category)
        self.cat_dtypes  = cat_dtypes       # {col: training CategoricalDtype}
        self.meta = meta if meta is not None else LinearRegression()

    def _stack(self, X):
        Xc = X.copy()
        for c, dt in self.cat_dtypes.items():
            Xc[c] = Xc[c].astype(dt)        # cast with the train category set
        preds = [m.predict(Xc if need_cat else X)
                 for _, m, need_cat in self.base_models]
        return np.column_stack(preds)

    def fit(self, X_blend, y_blend):
        self.meta.fit(self._stack(X_blend), y_blend)
        return self

    def predict(self, X):
        return self.meta.predict(self._stack(X))

print("Defined StackedGBM blender (meta-learner: LinearRegression).")

## Stage 9 — Training

Models are fit in family order. The **linear baselines** and the **RandomForest / ExtraTrees / HistGradientBoosting** trees fit on the plain integer training split (the linear pipelines scale + one-hot internally; HistGB carves its own early-stopping slice). The three **boosters** then train on the `log1p` target with early stopping against our validation split — CatBoost via `Pool`, XGBoost/LightGBM on **`category`-dtype copies** (built once below, with val/test cast to the *training* categories so the codes stay consistent). Finally the **`StackedGBM`** blender fits its linear meta-learner on the boosters' validation-set predictions. The `MODELS` dict is assembled in benchmark order and records the frame each model predicts on.

In [ ]:
import time, lightgbm as lgb

# Category-dtype copies for XGBoost / LightGBM. Build the TRAIN frame first to fix the
# category set, then cast val/test to the SAME categories so the integer->code mapping is
# consistent across splits (levels unseen in train become NaN = missing, handled natively).
def as_category(df, ref=None):
    out = df.copy()
    for c in CAT_FEATURES:
        out[c] = out[c].astype("category") if ref is None else out[c].astype(ref[c].dtype)
    return out
X_train_cat = as_category(X_train)
X_val_cat   = as_category(X_val,  ref=X_train_cat)
X_test_cat  = as_category(X_test, ref=X_train_cat)

# --- Linear baselines (own scale + one-hot pipeline; no validation / early stopping) ---
for nm, lm in [("Ridge", ridge_model), ("Linear", linear_model), ("Lasso", lasso_model)]:
    t0 = time.time(); lm.fit(X_train, y_train)
    print(f"{nm:20s} trained {time.time()-t0:6.1f}s")

# --- Bagging / HistGB baselines (plain integer frames; trees split on the codes) ---
for nm, tm in [("RandomForest", rf_model), ("ExtraTrees", et_model),
               ("HistGradientBoosting", hgb_model)]:
    t0 = time.time(); tm.fit(X_train, y_train)
    print(f"{nm:20s} trained {time.time()-t0:6.1f}s")

# --- CatBoost (native Pool) ---
train_pool = Pool(X_train, y_train, cat_features=CAT_FEATURES)
val_pool   = Pool(X_val,   y_val,   cat_features=CAT_FEATURES)
t0 = time.time(); cat_model.fit(train_pool, eval_set=val_pool)
print(f"{'CatBoost':20s} trained {time.time()-t0:6.1f}s | best_iter {cat_model.get_best_iteration()}")

# --- XGBoost ---
t0 = time.time(); xgb_model.fit(X_train_cat, y_train, eval_set=[(X_val_cat, y_val)], verbose=False)
print(f"{'XGBoost':20s} trained {time.time()-t0:6.1f}s | best_iter {xgb_model.best_iteration}")

# --- LightGBM ---
t0 = time.time()
lgb_model.fit(X_train_cat, y_train, eval_set=[(X_val_cat, y_val)], eval_metric="rmse",
              categorical_feature=CAT_FEATURES,
              callbacks=[lgb.early_stopping(100, verbose=False)])
print(f"{'LightGBM':20s} trained {time.time()-t0:6.1f}s | best_iter {lgb_model.best_iteration_}")

# --- Stacking ensemble: blend the 3 fitted boosters on the held-out validation split ---
cat_dtypes  = {c: X_train_cat[c].dtype for c in CAT_FEATURES}
stack_model = StackedGBM(
    base_models=[("XGBoost", xgb_model, True),
                 ("LightGBM", lgb_model, True),
                 ("CatBoost", cat_model, False)],
    cat_dtypes=cat_dtypes, meta=LinearRegression())
t0 = time.time(); stack_model.fit(X_val, y_val)   # meta learns weights on out-of-base-sample preds
w = stack_model.meta.coef_
print(f"{'Stacking(GBMs)':20s} trained {time.time()-t0:6.1f}s | meta weights "
      f"XGB/LGBM/Cat = {w[0]:+.2f}/{w[1]:+.2f}/{w[2]:+.2f}")

# Benchmark order: linear -> original boosters -> bagging/HistGB -> stacking (capstone).
# Each entry maps a model to the frames it predicts on in Stage 11.
MODELS = {
    "Ridge":    (ridge_model,  X_train,     X_test),
    "Linear":   (linear_model, X_train,     X_test),
    "Lasso":    (lasso_model,  X_train,     X_test),
    "XGBoost":  (xgb_model,    X_train_cat, X_test_cat),
    "LightGBM": (lgb_model,    X_train_cat, X_test_cat),
    "CatBoost": (cat_model,    X_train,     X_test),
    "RandomForest":         (rf_model,    X_train, X_test),
    "ExtraTrees":           (et_model,    X_train, X_test),
    "HistGradientBoosting": (hgb_model,   X_train, X_test),
    "Stacking(GBMs)":       (stack_model, X_train, X_test),
}

## Stage 10 — Learning Curve Visualisation

Left: CatBoost **train vs. validation** RMSE (the template's over/underfit diagnosis — a widening gap signals overfitting). Right: **validation RMSE for the three boosted-tree algorithms** overlaid, so their convergence and early-stopping points are directly comparable. (The linear baselines fit in closed form / coordinate descent with no boosting iterations, so they have no learning curve.) All curves are on the `log1p` target axis.

In [ ]:
ev_cat = cat_model.get_evals_result()
ev_xgb = xgb_model.evals_result()
ev_lgb = lgb_model.evals_result_

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Left: CatBoost train vs validation
ax[0].plot(ev_cat["learn"]["RMSE"], label="Train")
ax[0].plot(ev_cat["validation"]["RMSE"], label="Validation")
ax[0].axvline(cat_model.get_best_iteration(), ls="--", c="grey", label="best iter")
ax[0].set_title("CatBoost — train vs validation")
ax[0].set_xlabel("Iteration"); ax[0].set_ylabel("RMSE (log1p target)")
ax[0].legend(); ax[0].grid()

# Right: validation RMSE overlay across the three algorithms
ax[1].plot(ev_cat["validation"]["RMSE"],   label="CatBoost")
ax[1].plot(ev_xgb["validation_0"]["rmse"], label="XGBoost")
ax[1].plot(ev_lgb["valid_0"]["rmse"],      label="LightGBM")
ax[1].set_title("Validation RMSE by algorithm")
ax[1].set_xlabel("Iteration"); ax[1].set_ylabel("RMSE (log1p target)")
ax[1].legend(); ax[1].grid()

plt.tight_layout(); plt.show()

## Stage 11 — Evaluation on Training & Test Sets

Score every model on **both** train and test so the train-vs-test gap is visible. Predictions are made on the `log1p` axis, inverted with `expm1`, then multiplied by `WAGP_TO_DOLLARS = 1e6` to report **MAE / RMSE in dollars** (the extractor stored `WAGP` as adjusted dollars / 1e6). `R²` / Explained Variance are scale-free. The printed rows match the Experiment-Log format; `results_df` collects them.

In [ ]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             explained_variance_score, r2_score)

WAGP_TO_DOLLARS = 1_000_000
MODS = "High-card grouping: semantic rollup + rare-level pooling + Log-on-Target"

rows = []
print("| Algorithm | Modifications | Train MAE | Train RMSE | Train R2 | Test MAE | Test RMSE | Test R2 |")
print("|---|---|---|---|---|---|---|---|")
for name, (m, X_tr, X_te) in MODELS.items():           # MODELS is in benchmark order
    p_tr = np.expm1(m.predict(X_tr)) * WAGP_TO_DOLLARS
    p_te = np.expm1(m.predict(X_te)) * WAGP_TO_DOLLARS
    yt_tr = y_train_raw.values * WAGP_TO_DOLLARS
    yt_te = y_test_raw.values  * WAGP_TO_DOLLARS
    row = {
        "Algorithm": name, "Modifications": MODS,
        "Train MAE":  mean_absolute_error(yt_tr, p_tr),
        "Train RMSE": mean_squared_error(yt_tr, p_tr) ** 0.5,
        "Train R2":   r2_score(yt_tr, p_tr),
        "Test MAE":   mean_absolute_error(yt_te, p_te),
        "Test RMSE":  mean_squared_error(yt_te, p_te) ** 0.5,
        "Test R2":    r2_score(yt_te, p_te),
    }
    rows.append(row)
    print(f"| {name} | {MODS} | {row['Train MAE']:,.2f} | {row['Train RMSE']:,.2f} | "
          f"{row['Train R2']:.2f} | {row['Test MAE']:,.2f} | {row['Test RMSE']:,.2f} | {row['Test R2']:.2f} |")

# Keep rows in the benchmark order (Ridge -> Linear -> Lasso -> XGBoost -> LightGBM -> CatBoost);
# the best model is picked by Test R2 in Stage 12 via idxmax, not by row position.
results_df = pd.DataFrame(rows)
results_df

In [ ]:
# Feature importance per TREE model (normalised to sum=1 so the three are comparable).
# The linear baselines are excluded here: their "importance" lives in hundreds of
# post-one-hot coefficients, not the 12 raw features, so they aren't directly comparable.
TREE_MODELS = {k: MODELS[k] for k in ("XGBoost", "LightGBM", "CatBoost")}
fig, ax = plt.subplots(1, 3, figsize=(17, 6), sharey=True)
for a, (name, (m, _, _)) in zip(ax, TREE_MODELS.items()):
    imp = pd.Series(m.get_feature_importance() if name == "CatBoost"
                    else m.feature_importances_, index=list(X_train.columns))
    (imp / imp.sum()).sort_values().plot(kind="barh", ax=a)
    a.set_title(f"{name} feature importance")
plt.tight_layout(); plt.show()

## Stage 12 — Model Saving & Inference

Persist **all ten** fitted models (three linear pipelines, three boosters, RandomForest/ExtraTrees/HistGB, and the stacking blender) plus the shared preprocessing metadata (feature list, categorical columns, target transform, dollar scale) so inference reproduces the exact pipeline (saved as `acs2024_income_models_baseline.pkl`). The best model by Test R² is used for the sanity-check prediction.

> **Note:** the `Stacking(GBMs)` entry is a `StackedGBM` instance; reloading the bundle in a *fresh* process needs that class definition in scope (re-run Stage 8, or move the class into an importable module before deploying).

In [ ]:
# import joblib

# best_name = results_df.loc[results_df["Test R2"].idxmax(), "Algorithm"]
# bundle = {
#     "models": {n: t[0] for n, t in MODELS.items()},
#     "best_model": best_name,
#     "features": list(X_train.columns),
#     "cat_features": CAT_FEATURES,
#     "target": TARGET_COLUMN,
#     "target_transform": "log1p / expm1",
#     "wagp_to_dollars": WAGP_TO_DOLLARS,
#     "generated_features": GENERATED_COLS,
#     "group_map": GROUP_MAP,            # raw high-card col -> grouped col
#     "occp_bounds": OCCP_BOUNDS, "indp_bounds": INDP_BOUNDS, "pobp_bounds": POBP_BOUNDS,
#     "st_region": ST_REGION,
#     "rare_pool_min_count": MIN_COUNT, "rare_pool_other": POOL_OTHER, "pooled_levels": pooled,
#     "survey_year": "2024",
#     "results": results_df,
# }
# joblib.dump(bundle, "acs2024_income_models_exp3.pkl")
# print("Saved acs2024_income_models_exp3.pkl | best model:", best_name)

# # Reload and predict on a sample with the best model (invert log1p, x1e6 -> dollars).
# loaded = joblib.load("acs2024_income_models_exp3.pkl")
# m, X_tr, X_te = MODELS[loaded["best_model"]]
# scale = loaded["wagp_to_dollars"]
# sample_pred = np.expm1(m.predict(X_te[:5])) * scale
# print("\nGround-truth WAGP:", [f"${v:,.0f}" for v in y_test_raw.values[:5] * scale])
# print("Predicted   WAGP:", [f"${v:,.0f}" for v in sample_pred])

---
## Experiment Log — EXP3 (high-cardinality grouping: semantic rollup + rare-level pooling)

Ten regressors on the **grouped** feature set: `OCCP`/`INDP`/`POBP`/`ST` (1,055 levels) replaced by `OCCP_grp`/`INDP_grp`/`POBP_grp`/`ST_grp` (50 levels), then sparse `RELSHIPP`/`HISP` levels pooled into `-999`. MAE/RMSE in dollars; R² scale-free. Best Test R² in **bold**.

| Algorithm | Modifications | Train MAE | Train RMSE | Train R2 | Test MAE | Test RMSE | Test R2 |
|---|---|---|---|---|---|---|---|
| Ridge | exp3: rollup + rare-pool + Log-on-Target | 35,176.04 | 67,786.55 | 0.35 | 34,931.39 | 66,510.79 | 0.35 |
| Linear | exp3: rollup + rare-pool + Log-on-Target | 35,176.17 | 67,786.32 | 0.35 | 34,931.10 | 66,510.59 | 0.35 |
| Lasso | exp3: rollup + rare-pool + Log-on-Target | 35,058.93 | 68,050.82 | 0.34 | 34,789.60 | 66,735.51 | 0.35 |
| XGBoost | exp3: rollup + rare-pool + Log-on-Target | 28,337.10 | 58,285.92 | 0.52 | 29,993.39 | 61,160.74 | 0.45 |
| LightGBM | exp3: rollup + rare-pool + Log-on-Target | 28,926.77 | 59,325.41 | 0.50 | 30,013.18 | 61,184.97 | 0.45 |
| CatBoost | exp3: rollup + rare-pool + Log-on-Target | 29,685.09 | 60,776.30 | 0.47 | 30,116.60 | 61,303.11 | 0.45 |
| RandomForest | exp3: rollup + rare-pool + Log-on-Target | 24,192.92 | 51,522.52 | 0.62 | 31,234.51 | 62,759.13 | 0.43 |
| ExtraTrees | exp3: rollup + rare-pool + Log-on-Target | 23,734.64 | 50,708.80 | 0.63 | 30,878.76 | 62,610.40 | 0.43 |
| HistGradientBoosting | exp3: rollup + rare-pool + Log-on-Target | 29,701.03 | 60,692.47 | 0.48 | 30,295.50 | 61,487.44 | 0.45 |
| **Stacking(GBMs)** | exp3: rollup + rare-pool + Log-on-Target | 28,868.71 | 59,529.53 | 0.50 | **29,836.88** | **61,112.20** | **0.46** |

> Live from **Stage 11**; saved run: CatBoost best_iter 1997, XGBoost 243, LightGBM 533; stack meta weights XGB/LGBM/Cat ≈ +0.26/+0.36/+0.38. **Best: Stacking(GBMs) — Test R² 0.4551, MAE \$29,837.** Rare-pooling touched only `RELSHIPP` and `HISP` — the rollup had already made the high-card groups dense.

### Exp3 vs. baseline (raw codes) and exp2 (current best), Test R²

| Model | Baseline | Exp2 (best) | **Exp3** | Δ vs base |
|---|---|---|---|---|
| Stacking(GBMs) | 0.5079 | **0.5085** | 0.4551 | **−0.053** |
| LightGBM | 0.5048 | 0.5054 | 0.4538 | −0.051 |
| CatBoost | 0.4966 | 0.5034 | 0.4517 | −0.045 |
| XGBoost | 0.4952 | 0.4961 | 0.4542 | −0.041 |
| HistGradientBoosting | 0.4887 | 0.4996 | 0.4484 | −0.040 |
| RandomForest | 0.4643 | 0.4877 | 0.4254 | −0.039 |
| ExtraTrees | 0.4520 | 0.4908 | 0.4281 | −0.024 |
| Ridge / Linear | 0.4264 | 0.4283 | 0.3546 | −0.072 |
| Lasso | 0.3788 | 0.4131 | 0.3502 | −0.029 |

**Headline:** grouping is a **net loss for every model family** — Test R² falls ~0.02–0.07 across the board and the best score drops from **0.5085 (exp2) to 0.4551**. Coarsening throws away real signal.

**Why — and why it's the mirror image of exp2:**
- **Boosters lose ~0.04–0.05.** CatBoost/XGBoost/LightGBM read occupation- and industry-level wage structure straight from the 525/257 raw codes; rolling them up to 24/16 groups erases the within-group spread (a software developer and a line cook only share a "major group" because we defined the buckets coarsely). CatBoost's now-tight train/test gap (0.47/0.45) shows it is neither over- nor under-fitting — it has simply hit a **lower ceiling** because the feature can no longer separate those jobs.
- **The linear models lose the MOST (Ridge −0.072), refuting the pre-run hypothesis.** One-hotting 50 grouped levels instead of ~1,055 raw dummies was *expected* to help Ridge/Lasso; it did the opposite. Those ~1,055 occupation/industry/birthplace/state dummies were exactly how the linear models encoded "this specific occupation pays X," and Ridge's L2 penalty already handled the width. Collapsing them removed that capacity.
- **Forests lose least** (ExtraTrees −0.024): they read the integer codes as ordinal anyway, so they got limited value from the fine codes; grouping mostly just trims their over-splitting.
- **Rare-level pooling was nearly inert.** After the semantic rollup, only `RELSHIPP`/`HISP` had sub-30-row levels, so the low-sample lever changed almost nothing — the sparsity lived entirely in the high-card codes, which the rollup had already dissolved.

**Takeaways:**
- This is the **complementary result to exp2**, and the two agree: the wage signal lives in the *granular* high-cardinality codes. Exp2 *added* numeric views of them and lifted the laggards; exp3 *removed* their granularity and lowered everyone. **Keep the raw codes.**
- Grouping/coarsening is the **wrong lever for accuracy here.** Its real value is elsewhere — a smaller/faster model, interpretable coefficients, or robustness to occupations unseen at training time. If a coarse view is wanted, **add `*_grp` columns *alongside* the raw codes** (augment, like exp2) rather than replacing them.
- The field still sits below the ~0.51 ceiling, now approached from the wrong side. Confirms again that the next gains come from **richer raw predictors** (PUMA geography, finer occupation detail, hours×industry, modelling the top-coded tail, `PWGTP` weights) — not from re-bucketing what we already have.

*Notebook scaffold generated from `ML_Model_Building_Template.md`, wired to `acs2024_income.csv`.*